In [5]:
import pandas as pd
import numpy as np
import re
import glob

In [9]:
df = pd.read_csv("../raw_data/amazon_india_2015.csv")

In [ ]:
df.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2015_00000001,2015-01-25,CUST_2015_00003884,PROD_000021,Samsung Galaxy S6 16GB Black,Electronics,Smartphones,Samsung,123614.29,27.91,...,True,Republic Day Sale,5.0,Delivered,1,2015,1,0.19,True,4.7
1,TXN_2015_00000002,2015-01-05,CUST_2015_00011709,PROD_000055,OnePlus OnePlus 2 16GB White,Electronics,Smartphones,OnePlus,54731.86,0.00,...,False,NaN,4.5,Delivered,1,2015,1,0.20,True,4.1
2,TXN_2015_00000003,2015-01-24,CUST_2015_00004782,PROD_000039,Samsung Galaxy Note 5 64GB Black,Electronics,Smartphones,Samsung,97644.25,46.93,...,True,Republic Day Sale,NaN,Delivered,1,2015,1,0.17,True,3.3
3,TXN_2015_00000004,2015-01-28,CUST_2015_00008105,PROD_000085,Motorola Moto G (3rd Gen) 16GB Black,Electronics,Smartphones,Motorola,"21,947.26",0.00,...,False,NaN,3.0,Delivered,1,2015,1,0.22,True,3.5
4,TXN_2015_00000005,2015-01-31,CUST_2015_00002955,PROD_000055,OnePlus OnePlus 2 16GB White,Electronics,Smartphones,OnePlus,54731.86,0.00,...,FALSE,NaN,4.0,Delivered,1,2015,1,0.20,True,4.1


In [ ]:
df["delivery_charges"].isna().sum(), len(df)

(np.int64(2654), 33165)

In [ ]:
df["delivery_charges"].describe()

count    30511.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: delivery_charges, dtype: float64

In [ ]:
df.drop(columns=["delivery_charges"], inplace=True)

In [ ]:
df.columns

Index(['transaction_id', 'order_date', 'customer_id', 'product_id',
       'product_name', 'category', 'subcategory', 'brand',
       'original_price_inr', 'discount_percent', 'discounted_price_inr',
       'quantity', 'subtotal_inr', 'final_amount_inr', 'customer_city',
       'customer_state', 'customer_tier', 'customer_spending_tier',
       'customer_age_group', 'payment_method', 'delivery_days',
       'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name',
       'customer_rating', 'return_status', 'order_month', 'order_year',
       'order_quarter', 'product_weight_kg', 'is_prime_eligible',
       'product_rating'],
      dtype='object')

In [ ]:
df.shape

(33165, 33)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33165 entries, 0 to 33164
Data columns (total 33 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   transaction_id          33165 non-null  object 
 1   order_date              33165 non-null  object 
 2   customer_id             33165 non-null  object 
 3   product_id              33165 non-null  object 
 4   product_name            33165 non-null  object 
 5   category                33165 non-null  object 
 6   subcategory             33165 non-null  object 
 7   brand                   33165 non-null  object 
 8   original_price_inr      33165 non-null  object 
 9   discount_percent        33165 non-null  float64
 10  discounted_price_inr    33165 non-null  float64
 11  quantity                33165 non-null  int64  
 12  subtotal_inr            33165 non-null  float64
 13  final_amount_inr        33165 non-null  float64
 14  customer_city           33165 non-null

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [ ]:
df["order_date"].head(20)

0     2015-01-25
1     2015-01-05
2     2015-01-24
3     2015-01-28
4     2015-01-31
5     2015-01-04
6     2015-01-27
7     2015-01-08
8     2015-01-18
9     2015-01-03
10    2015-01-25
11    2015-01-15
12    2015-01-26
13    2015-01-11
14    2015-01-08
15    2015-01-07
16    2015-01-25
17    2015-01-08
18    2015-01-06
19    2015-01-11
Name: order_date, dtype: object

In [ ]:
df["order_date"] = (
    df["order_date"]
    .str.replace(" ", "", regex=False)
    .str.replace("/", "-", regex=False)
)

parts = df["order_date"].str.split("-", expand=True)

year_last = parts[2].str.len() == 4

df.loc[year_last, "order_date"] = (
    parts[2] + "-" + parts[0] + "-" + parts[1]
)

parts = df["order_date"].str.split("-", expand=True)

mask = parts[1].astype(int) > 12

df.loc[mask, "order_date"] = (
    parts[0] + "-" + parts[2] + "-" + parts[1]
)

df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

In [ ]:
df["order_date"].min(), df["order_date"].max()

(Timestamp('2015-01-01 00:00:00'), Timestamp('2015-12-31 00:00:00'))

In [ ]:
df["order_date"].isna().sum()

np.int64(0)

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees. 


In [ ]:
df["original_price_inr"] = df["original_price_inr"].astype(str)

df["original_price_inr"] = df["original_price_inr"].str.replace("₹", "", regex=False)

df["original_price_inr"] = df["original_price_inr"].str.replace(",", "", regex=False)

df["original_price_inr"] = pd.to_numeric(df["original_price_inr"], errors="coerce")

In [ ]:
df["original_price_inr"].unique()[:20]

array([123614.29,  54731.86,  97644.25,  21947.26, 131194.65,  86987.64,
        32169.01,  40264.16,  88664.85,  73967.02,  72564.1 , 209875.55,
        45363.47,  23075.71, 114096.63,  69584.33,  38884.19,  46234.09,
        15065.01, 164166.84])

In [ ]:
df["original_price_inr"].dtypes

dtype('float64')

Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.


In [ ]:
df["customer_rating"] = df["customer_rating"].astype(str)

df["customer_rating"] = df["customer_rating"].str.replace(" stars", "", regex=False)

df["customer_rating"] = df["customer_rating"].str.split("/").str[0]

df["customer_rating"] = pd.to_numeric(df["customer_rating"], errors="coerce")

In [ ]:
df["customer_rating"].describe()

count    23196.000000
mean         4.321564
std          0.574197
min          3.000000
25%          4.000000
50%          4.500000
75%          5.000000
max          5.000000
Name: customer_rating, dtype: float64

In [ ]:
df["customer_rating"].value_counts().head(10)

customer_rating
4.5    7590
5.0    6197
4.0    5713
3.5    2326
3.0    1370
Name: count, dtype: int64

In [ ]:
df["customer_rating"].isna().sum()

np.int64(9969)

In [ ]:
df["customer_rating"].value_counts(dropna=False)

customer_rating
NaN    9969
4.5    7590
5.0    6197
4.0    5713
3.5    2326
3.0    1370
Name: count, dtype: int64

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.


In [ ]:
df["customer_city"] = df["customer_city"].str.strip().str.lower()

In [ ]:
df["customer_city"].unique()

array(['mumbai', 'allahabad', 'kolkata', 'ludhiana', 'delhi', 'lucknow',
       'jaipur', 'bhubaneswar', 'ahmedabad', 'bangalore', 'pune', 'kochi',
       'chennai', 'nagpur', 'visakhapatnam', 'mumba', 'gorakhpur',
       'bombay', 'kanpur', 'chandigarh', 'hyderabad', 'aligarh', 'indore',
       'patna', 'meerut', 'coimbatore', 'vadodara', 'saharanpur',
       'bareilly', 'moradabad', 'new delhi', 'surat', 'banglore',
       'madras', 'chenai', 'delhi ncr', 'varanasi', 'bengalore',
       'bengaluru', 'calcutta'], dtype=object)

In [ ]:
city_map = {
    "bangalore": "bengaluru",
    "banglore": "bengaluru",
    "bengalore": "bengaluru",
    "bengaluru": "bengaluru",

    "bombay": "mumbai",
    "mumba": "mumbai",
    "mumbai": "mumbai",

    "madras": "chennai",
    "chenai": "chennai",
    "chennai": "chennai",

    "new delhi": "delhi",
    "delhi ncr": "delhi",
    "delhi": "delhi",

    "calcutta": "kolkata",
    "kolkata": "kolkata"
}

In [ ]:
df["customer_city"] = df["customer_city"].replace(city_map)

In [ ]:
df["customer_city"] = df["customer_city"].str.title()

In [ ]:
df["customer_city"].value_counts().head(20)

customer_city
Mumbai           5391
Delhi            4792
Bengaluru        3847
Chennai          3375
Kolkata          2524
Hyderabad        1752
Pune             1444
Ahmedabad        1104
Surat             909
Jaipur            828
Nagpur            819
Kanpur            751
Lucknow           718
Indore            662
Coimbatore        547
Kochi             491
Chandigarh        448
Bhubaneswar       418
Visakhapatnam     404
Patna             379
Name: count, dtype: int64

In [ ]:
df["customer_city"].unique()

array(['Mumbai', 'Allahabad', 'Kolkata', 'Ludhiana', 'Delhi', 'Lucknow',
       'Jaipur', 'Bhubaneswar', 'Ahmedabad', 'Bengaluru', 'Pune', 'Kochi',
       'Chennai', 'Nagpur', 'Visakhapatnam', 'Gorakhpur', 'Kanpur',
       'Chandigarh', 'Hyderabad', 'Aligarh', 'Indore', 'Patna', 'Meerut',
       'Coimbatore', 'Vadodara', 'Saharanpur', 'Bareilly', 'Moradabad',
       'Surat', 'Varanasi'], dtype=object)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [ ]:
bool_candidates = []

bool_values = {"true","false","yes","no","y","n","1","0"}

for col in df.columns:
    vals = set(df[col].astype(str).str.lower().dropna().unique())
    
    if vals & bool_values:
        bool_candidates.append(col)

bool_candidates

['quantity',
 'delivery_days',
 'is_prime_member',
 'is_festival_sale',
 'order_month',
 'order_quarter',
 'is_prime_eligible']

In [ ]:
boolean_cols = ["is_prime_member", "is_prime_eligible", "is_festival_sale"]

bool_map = {
    "true": True,
    "false": False,
    "yes": True,
    "no": False,
    "y": True,
    "n": False,
    "1": True,
    "0": False
}

for col in boolean_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(bool_map)
    )

In [ ]:
df[boolean_cols].value_counts(dropna=False)

is_prime_member  is_prime_eligible  is_festival_sale
False            True               False               17452
                                    True                 8228
                 False              False                5129
                                    True                 2356
Name: count, dtype: int64

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.


In [7]:
df["category"].value_counts().head(20)

category
Electronics                  33085
ELECTRONICS                     29
Electronic                      25
Electronics & Accessories       16
Electronicss                    10
Name: count, dtype: int64

In [14]:
df["category"] = df["category"].str.strip().str.lower()


In [ ]:
category_map = {
    "electronic": "electronics",
    "electronics": "electronics",
    "electronics & accessories": "electronics",
    "electronicss": "electronics"
}

In [17]:
df["category"] = df["category"].replace(category_map)
df["category"] = df["category"].str.title()

In [19]:
df["category"].value_counts()

category
Electronics    33165
Name: count, dtype: int64

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [ ]:
df["delivery_days"].unique()

array(['6', '4', '3', '5', '7', 'Express', '0', '-1', 'Same Day',
       '1-2 days', '15'], dtype=object)

In [ ]:
df["delivery_days"] = df["delivery_days"].astype(str).str.strip().str.lower()

In [ ]:
df["delivery_days"] = df["delivery_days"].replace({
    "same day": "0",
    "express": "1"
})

In [ ]:
df["delivery_days"] = df["delivery_days"].str.extract(r"(-?\d+)")

In [ ]:
df["delivery_days"] = pd.to_numeric(df["delivery_days"], errors="coerce")

In [ ]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = None

In [ ]:
df["delivery_days"].unique()

array([ 6.,  4.,  3.,  5.,  7.,  1.,  0., nan, 15.])

In [ ]:
df["delivery_days"].isnull().sum()

np.int64(208)

In [ ]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = np.nan

df["delivery_days"] = df["delivery_days"].fillna(df["delivery_days"].median())

In [ ]:
df["delivery_days"].describe()

count    32175.000000
mean         4.319472
std          1.340621
min          0.000000
25%          3.000000
50%          4.000000
75%          5.000000
max         15.000000
Name: delivery_days, dtype: float64

In [ ]:
df["delivery_days"].isna().sum()

np.int64(0)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [ ]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [ ]:
duplicates.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
279,TXN_2015_00000280,2015-01-29,CUST_2015_00010848,PROD_000027,Samsung Galaxy S6 16GB Blue,Electronics,Smartphones,Samsung,NaN,21.03,...,False,NaN,4.0,Delivered,1,2015,1,0.18,True,3.4
814,TXN_2015_00000815,2015-01-31,CUST_2015_00001928,PROD_000059,OnePlus OnePlus X 32GB Black,Electronics,Smartphones,OnePlus,57182.39,0.00,...,False,NaN,5.0,Delivered,1,2015,1,0.20,True,3.8
941,TXN_2015_00000942,2015-01-03,CUST_2015_00007653,PROD_000075,Xiaomi Redmi 2 64GB White,Electronics,Smartphones,Xiaomi,35581.67,0.00,...,False,NaN,NaN,Delivered,1,2015,1,0.22,False,3.6
996,TXN_2015_00000997,2015-01-28,CUST_2015_00003254,PROD_000032,Samsung Galaxy S6 Edge 16GB White,Electronics,Smartphones,Samsung,97905.92,0.00,...,False,NaN,4.0,Delivered,1,2015,1,0.20,True,4.3
1390,TXN_2015_00001391,2015-01-15,CUST_2015_00005080,PROD_001640,MSI Aspire 4GB RAM Black,Electronics,Laptops,MSI,69584.33,12.76,...,False,NaN,4.5,Returned,1,2015,1,2.66,True,3.2


In [ ]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [ ]:
duplicates.shape

(320, 33)

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df = df.drop_duplicates()

In [ ]:
duplicates.sort_values(
    ["customer_id","product_id","order_date"]
).head(10)

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
1851,TXN_2015_00001852,2015-01-23,CUST_2015_00000037,PROD_000053,OnePlus OnePlus 2 32GB Black,Electronics,Smartphones,OnePlus,102649.14,19.30,...,True,Republic Day Sale,5.0,Delivered,1,2015,1,0.20,True,4.6
33077,TXN_2015_00001852_DUP,2015-01-23,CUST_2015_00000037,PROD_000053,OnePlus OnePlus 2 32GB Black,Electronics,Smartphones,OnePlus,102649.14,19.30,...,True,Republic Day Sale,5.0,Delivered,1,2015,1,0.20,True,4.6
11861,TXN_2015_00011862,2015-05-23,CUST_2015_00000173,PROD_001820,OnePlus Neckband Premium,Electronics,Audio,OnePlus,19375.09,13.40,...,False,NaN,3.5,Delivered,5,2015,2,0.42,False,3.5
33063,TXN_2015_00011862_DUP,2015-05-23,CUST_2015_00000173,PROD_001820,OnePlus Neckband Premium,Electronics,Audio,OnePlus,19375.09,13.40,...,False,NaN,3.5,Delivered,5,2015,2,0.42,False,3.5
18161,TXN_2015_00018162,2015-08-25,CUST_2015_00000265,PROD_000071,Xiaomi Redmi 2 32GB Black,Electronics,Smartphones,Xiaomi,44203.12,0.00,...,False,NaN,4.5,Delivered,8,2015,3,0.20,True,3.7
33135,TXN_2015_00018162_DUP,2015-08-25,CUST_2015_00000265,PROD_000071,Xiaomi Redmi 2 32GB Black,Electronics,Smartphones,Xiaomi,44203.12,0.00,...,False,NaN,4.5,Delivered,8,2015,3,0.20,True,3.7
14926,TXN_2015_00014927,2015-07-27,CUST_2015_00000326,PROD_000036,Samsung Galaxy S6 Edge 16GB Gold,Electronics,Smartphones,Samsung,88872.68,0.00,...,False,NaN,NaN,Delivered,7,2015,3,0.16,False,3.9
33164,TXN_2015_00014927_DUP,2015-07-27,CUST_2015_00000326,PROD_000036,Samsung Galaxy S6 Edge 16GB Gold,Electronics,Smartphones,Samsung,88872.68,0.00,...,False,NaN,NaN,Delivered,7,2015,3,0.16,False,3.9
21161,TXN_2015_00021162,2015-09-08,CUST_2015_00000433,PROD_000071,Xiaomi Redmi 2 32GB Black,Electronics,Smartphones,Xiaomi,44203.12,7.81,...,False,NaN,5.0,Delivered,9,2015,3,0.20,True,3.7
33048,TXN_2015_00021162_DUP,2015-09-08,CUST_2015_00000433,PROD_000071,Xiaomi Redmi 2 32GB Black,Electronics,Smartphones,Xiaomi,44203.12,7.81,...,False,NaN,5.0,Delivered,9,2015,3,0.20,True,3.7


In [ ]:
duplicates.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().sort_values(ascending=False).head(10)

customer_id         product_id   order_date  original_price_inr
CUST_2015_00000037  PROD_000053  2015-01-23  102649.14             2
CUST_2015_00000173  PROD_001820  2015-05-23  19375.09              2
CUST_2015_00000265  PROD_000071  2015-08-25  44203.12              2
CUST_2015_00000326  PROD_000036  2015-07-27  88872.68              2
CUST_2015_00000433  PROD_000071  2015-09-08  44203.12              2
CUST_2015_00000445  PROD_000072  2015-09-14  20937.46              2
CUST_2015_00000449  PROD_000084  2015-10-18  30990.65              2
CUST_2015_00000485  PROD_001713  2015-12-16  67444.91              2
CUST_2015_00000492  PROD_001671  2015-02-08  107387.37             2
CUST_2015_00000523  PROD_000045  2015-12-09  41408.67              2
dtype: int64

In [ ]:
dup_groups = df.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().reset_index(name="count")

dup_groups = dup_groups[dup_groups["count"] > 1]

In [ ]:
dup_rows = df.merge(
    dup_groups,
    on=["customer_id","product_id","order_date","original_price_inr"],
    how="inner"
)

In [ ]:
dup_rows[["customer_id","product_id","quantity","count"]].head()

,customer_id,product_id,quantity,count
0,CUST_2015_00001928,PROD_000059,1,2
1,CUST_2015_00007653,PROD_000075,1,2
2,CUST_2015_00003254,PROD_000032,1,2
3,CUST_2015_00005080,PROD_001640,1,2
4,CUST_2015_00002994,PROD_000063,1,2


In [ ]:
df_clean = df.drop_duplicates(
    subset=["customer_id","product_id","order_date","original_price_inr"],
    keep="first"
)

In [ ]:
df_clean.duplicated(
    subset=["customer_id","product_id","order_date","original_price_inr"]
).sum()

np.int64(0)

In [ ]:
df["transaction_id"].duplicated().sum()

np.int64(0)

In [ ]:
df = df.drop_duplicates(subset="transaction_id", keep="first")

In [ ]:
df[df["transaction_id"]=="TXN_2015_00000280"]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
279,TXN_2015_00000280,2015-01-29,CUST_2015_00010848,PROD_000027,Samsung Galaxy S6 16GB Blue,Electronics,Smartphones,Samsung,NaN,21.03,...,False,NaN,4.0,Delivered,1,2015,1,0.18,True,3.4


Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [ ]:
print(f"NaN in subtotal_inr:         {df['subtotal_inr'].isna().sum()}")
print(f"NaN in discounted_price_inr: {df['discounted_price_inr'].isna().sum()}")
print(f"NaN in quantity:             {df['quantity'].isna().sum()}")
print(f"NaN in discount_percent:     {df['discount_percent'].isna().sum()}")

NaN in subtotal_inr:         0
NaN in discounted_price_inr: 0
NaN in quantity:             0
NaN in discount_percent:     0


In [ ]:
print(f"NaN in original_price_inr: {df['original_price_inr'].isna().sum()}")

# Check the actual NaN rows
nan_rows = df[df["discounted_price_inr"].isna()]
print(nan_rows[["product_name", "original_price_inr", "discount_percent", "discounted_price_inr"]].head(10))

NaN in original_price_inr: 987
Empty DataFrame
Columns: [product_name, original_price_inr, discount_percent, discounted_price_inr]
Index: []


In [ ]:
# Check which subcategories these NaN prices belong to
print(df[df["original_price_inr"].isna()]["subcategory"].value_counts())

subcategory
Smartphones           611
Smart Watch           110
Tablets               106
Laptops               102
Audio                  36
TV & Entertainment     22
Name: count, dtype: int64


In [ ]:
print(df[df["original_price_inr"].isna()].head(10).to_string())

        transaction_id order_date         customer_id   product_id                          product_name     category  subcategory     brand  original_price_inr  discount_percent  discounted_price_inr  quantity  subtotal_inr  final_amount_inr customer_city customer_state customer_tier customer_spending_tier customer_age_group payment_method  delivery_days delivery_type  is_prime_member  is_festival_sale      festival_name  customer_rating return_status  order_month  order_year  order_quarter  product_weight_kg  is_prime_eligible  product_rating
62   TXN_2015_00000063 2015-01-15  CUST_2015_00006127  PROD_001922                          Fitbit Watch  Electronics  Smart Watch    Fitbit                 NaN              0.00              47978.89         1      47978.89          47978.89       Kolkata    West Bengal         Metro               Standard              36-45    Credit Card            6.0      Standard            False             False                NaN              5.0      R

In [ ]:
# Drop rows with NaN original_price_inr
df = df.dropna(subset=["original_price_inr"]).copy()  # ← added .copy()

# Recalculate
df["discounted_price_inr"] = (df["original_price_inr"] * (1 - df["discount_percent"] / 100)).round(2)
df["subtotal_inr"] = (df["discounted_price_inr"] * df["quantity"]).round(2)
df["final_amount_inr"] = df["subtotal_inr"]




In [ ]:
# ── FIX 1: Negative prices ──────────────────────────
neg_mask = df["original_price_inr"] < 0
df.loc[neg_mask, "original_price_inr"] = df.loc[neg_mask, "original_price_inr"].abs()
print(f"Negative prices fixed: {neg_mask.sum()}")

# ── FIX 2: Outliers (IQR method - 2 passes) ─────────
subcategory_caps = {
    "Smart Watch":        50000,
    "Tablets":            80000,
    "Smartphones":        150000,
    "Laptops":            200000,
    "TV & Entertainment": 300000,
    "Audio":              50000,
}

for sub, cap in subcategory_caps.items():
    for pass_num in range(1, 3):
        mask = df["subcategory"] == sub
        prices = df.loc[mask, "original_price_inr"]

        Q1 = prices.quantile(0.25)
        Q3 = prices.quantile(0.75)
        IQR = Q3 - Q1
        upper_bound = min(Q3 + 1.5 * IQR, cap)  # cap as hard ceiling

        outlier_mask = mask & (df["original_price_inr"] > upper_bound)
        df.loc[outlier_mask, "original_price_inr"] = (
            df.loc[outlier_mask, "original_price_inr"] / 10
        ).round(2)

        print(f"{sub} (pass {pass_num}): {outlier_mask.sum()} outliers fixed (upper bound: ₹{upper_bound:,.2f})")

# ── FIX 3: Recalculate ───────────────────────────────
df["discounted_price_inr"] = (df["original_price_inr"] * (1 - df["discount_percent"] / 100)).round(2)
df["subtotal_inr"] = (df["discounted_price_inr"] * df["quantity"]).round(2)
df["final_amount_inr"] = df["subtotal_inr"]

# ── VERIFY ───────────────────────────────────────────
print(df.groupby("subcategory", observed=True)["original_price_inr"]
      .describe()[["min","max","mean","50%"]].round(2))
print(f"\nNaN in final_amount_inr:   {df['final_amount_inr'].isna().sum()}")
print(f"Negative prices remaining: {(df['original_price_inr'] < 0).sum()}")

for sub, cap in subcategory_caps.items():
    over = df[(df["subcategory"] == sub) &
              (df["original_price_inr"] > cap)]
    if len(over) > 0:
        print(f"\n{sub} (cap ₹{cap:,}): {len(over)} rows over")
        print(over[["product_name", "original_price_inr"]].drop_duplicates().head(5))
    else:
        print(f"\n{sub}: ✅ All within cap")

Negative prices fixed: 84
Smart Watch (pass 1): 16 outliers fixed (upper bound: ₹50,000.00)
Smart Watch (pass 2): 8 outliers fixed (upper bound: ₹50,000.00)
Tablets (pass 1): 16 outliers fixed (upper bound: ₹80,000.00)
Tablets (pass 2): 9 outliers fixed (upper bound: ₹80,000.00)
Smartphones (pass 1): 1646 outliers fixed (upper bound: ₹150,000.00)
Smartphones (pass 2): 56 outliers fixed (upper bound: ₹150,000.00)
Laptops (pass 1): 20 outliers fixed (upper bound: ₹200,000.00)
Laptops (pass 2): 9 outliers fixed (upper bound: ₹200,000.00)
TV & Entertainment (pass 1): 93 outliers fixed (upper bound: ₹157,867.54)
TV & Entertainment (pass 2): 4 outliers fixed (upper bound: ₹157,867.54)
Audio (pass 1): 2 outliers fixed (upper bound: ₹32,715.58)
Audio (pass 2): 1 outliers fixed (upper bound: ₹32,715.58)
                         min        max      mean        50%
subcategory                                                 
Audio                3795.01   23075.71  14774.86   16406.11
Laptops    

In [ ]:
outlier_indices = [19488, 25511, 29293]
df = df.drop(index=outlier_indices)
print(f"Rows remaining: {len(df)}")

KeyError: '[19488, 25511, 29293] not found in axis'

In [ ]:
df.groupby("subcategory", observed=True)["original_price_inr"].max().reset_index().sort_values("original_price_inr", ascending=False)

,subcategory,original_price_inr
1,Laptops,185009.58
4,TV & Entertainment,179795.49
3,Smartphones,148346.57
5,Tablets,77250.03
2,Smart Watch,47978.89
0,Audio,23075.71


Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 
'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [ ]:
# ── Standardize payment methods ────────────────────
payment_standardize = {
    # UPI variants
    "UPI": "UPI", "PhonePe": "UPI", "GooglePay": "UPI", "Google Pay": "UPI",
    "UPI/PhonePe": "UPI", "UPI/GooglePay": "UPI",

    # Credit Card variants
    "Credit Card": "Credit Card", "CREDIT_CARD": "Credit Card", "CC": "Credit Card",

    # Debit Card variants
    "Debit Card": "Debit Card", "DEBIT_CARD": "Debit Card", "DC": "Debit Card",

    # COD variants
    "Cash on Delivery": "COD", "COD": "COD", "C.O.D": "COD",

    # Others
    "Wallet": "Wallet",
    "Net Banking": "Net Banking",
    "BNPL": "BNPL"
}

df["payment_method"] = df["payment_method"].map(payment_standardize).fillna(df["payment_method"])

# ── Create categorical hierarchy ───────────────────
payment_category = {
    "UPI":          "Digital Payment",
    "Wallet":       "Digital Payment",
    "Net Banking":  "Digital Payment",
    "Credit Card":  "Card Payment",
    "Debit Card":   "Card Payment",
    "BNPL":         "Pay Later",
    "COD":          "Cash on Delivery"
}

df["payment_category"] = df["payment_method"].map(payment_category).astype("category")

# ── Verify ─────────────────────────────────────────
print(df["payment_method"].value_counts())
print(f"\nNaN in payment_category: {df['payment_category'].isna().sum()}")

payment_method
COD            24212
Credit Card     3886
Debit Card      2528
Net Banking     1552
Name: count, dtype: int64

NaN in payment_category: 0


Handling Nan - in customer age group

In [ ]:
df["customer_age_group"] = df["customer_age_group"].fillna("Unknown")

Checking for object columns

In [ ]:
df.select_dtypes(include="object").columns

Index(['transaction_id', 'customer_id', 'product_id', 'product_name',
       'category', 'subcategory', 'brand', 'customer_city', 'customer_state',
       'customer_tier', 'customer_spending_tier', 'customer_age_group',
       'payment_method', 'delivery_type', 'festival_name', 'return_status'],
      dtype='object')

In [ ]:
# ── Optimize memory: convert to category dtype ───────
cat_columns = ["category", "subcategory", "customer_tier", 
               "customer_spending_tier", "customer_age_group",
               "payment_method", "delivery_type", 
               "festival_name", "return_status"]

df[cat_columns] = df[cat_columns].astype("category")

# Verify
print(df[cat_columns].dtypes)
print(f"\nMemory usage after optimization:")
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")

category                  category
subcategory               category
customer_tier             category
customer_spending_tier    category
customer_age_group        category
payment_method            category
delivery_type             category
festival_name             category
return_status             category
dtype: object

Memory usage after optimization:
17.56376075744629 MB


In [ ]:
# NaN summary for all columns
nan_summary = df.isna().sum()
nan_summary = nan_summary[nan_summary > 0].sort_values(ascending=False)

print(f"Total columns with NaN: {len(nan_summary)}")
print(f"Total rows in dataset: {df.shape[0]}")
print(f"\nNaN counts and percentages:")
print(pd.DataFrame({
    "NaN Count": nan_summary,
    "Percentage": (nan_summary / df.shape[0] * 100).round(2)
}))

Total columns with NaN: 2
Total rows in dataset: 32175

NaN counts and percentages:
                 NaN Count  Percentage
festival_name        21914       68.11
customer_rating       9676       30.07


In [ ]:
df.to_csv("data_cleaning_2015.csv", index=False)
print("File saved successfully!")

File saved successfully!
